# 03 Test

This notebook runs the same prompt set against both direct Azure OpenAI deployments and the APIM gateway so you can compare responses side by side.

## Load Environment

In [ ]:
import json
import os
from pathlib import Path
from dotenv import load_dotenv

env_path = Path('../env/.env')
if not env_path.exists():
    raise RuntimeError('env/.env not found. Run 01_deploy_infra.ipynb first.')

load_dotenv(env_path)

AZURE_OPENAI_ENDPOINT = os.getenv('AZURE_OPENAI_ENDPOINT', '').rstrip('/')
AZURE_OPENAI_API_KEY = os.getenv('AZURE_OPENAI_API_KEY', '')
AZURE_OPENAI_DEPLOYMENT = os.getenv('AZURE_OPENAI_DEPLOYMENT', '')
AZURE_OPENAI_SECONDARY_DEPLOYMENT = os.getenv('AZURE_OPENAI_SECONDARY_DEPLOYMENT', '')
AZURE_OPENAI_API_VERSION = os.getenv('AZURE_OPENAI_API_VERSION', '2024-10-21')
APIM_ENDPOINT = os.getenv('APIM_ENDPOINT', '').rstrip('/')
APIM_PATH = os.getenv('APIM_CHAT_COMPLETIONS_PATH', '/openai/chat/completions')
APIM_SECONDARY_PATH = os.getenv('APIM_SECONDARY_CHAT_COMPLETIONS_PATH', '/openai51/chat/completions')
APIM_API_KEY = os.getenv('APIM_API_KEY', '')

print(f'Azure OpenAI endpoint: {AZURE_OPENAI_ENDPOINT}')
print(f'Primary deployment: {AZURE_OPENAI_DEPLOYMENT}')
print(f'Secondary deployment: {AZURE_OPENAI_SECONDARY_DEPLOYMENT}')
print(f'APIM endpoint: {APIM_ENDPOINT}')
print(f'APIM primary path: {APIM_PATH}')
print(f'APIM secondary path: {APIM_SECONDARY_PATH}')

## Test Each Target

In [ ]:
import requests

if not AZURE_OPENAI_API_KEY:
    raise RuntimeError('AZURE_OPENAI_API_KEY is not set in env/.env')

test_prompts = [
    'What is the capital of France?',
    'Explain machine learning in one sentence.',
    'Name three primary colors.'
]

results = []

for prompt_index, prompt in enumerate(test_prompts, start=1):
    print(f'\n===== Primary Deployment Prompt {prompt_index}: {prompt} =====')

    url = f"{AZURE_OPENAI_ENDPOINT}/openai/deployments/{AZURE_OPENAI_DEPLOYMENT}/chat/completions?api-version={AZURE_OPENAI_API_VERSION}"
    headers = {
        'api-key': AZURE_OPENAI_API_KEY,
        'Content-Type': 'application/json'
    }
    payload = {
        'messages': [
            {'role': 'system', 'content': 'You are a concise assistant.'},
            {'role': 'user', 'content': prompt}
        ],
        'max_tokens': 120
    }

    response = requests.post(url, headers=headers, json=payload, timeout=45)

    response_model = ''
    if response.status_code == 200:
        body = response.json()
        answer = body.get('choices', [{}])[0].get('message', {}).get('content', '')
        response_model = body.get('model', '')
        status = 'success'
    else:
        answer = response.text[:600]
        status = f"error_{response.status_code}"

    row = {
        'prompt_index': prompt_index,
        'query': prompt,
        'target': 'direct-primary',
        'label': f'direct:{AZURE_OPENAI_DEPLOYMENT}',
        'deployment': AZURE_OPENAI_DEPLOYMENT,
        'response': answer,
        'model': response_model,
        'status': status
    }
    results.append(row)

    print(f'Status: {status}')
    print(f'Deployment: {AZURE_OPENAI_DEPLOYMENT}')
    print(f"Model: {response_model if response_model else '[not returned]'}")
    print('Output:')
    print(answer.strip() if answer else '[empty response]')

## Test Direct Secondary Deployment

In [ ]:
import requests

if not AZURE_OPENAI_API_KEY:
    raise RuntimeError('AZURE_OPENAI_API_KEY is not set in env/.env')

for prompt_index, prompt in enumerate(test_prompts, start=1):
    print(f'\n===== Secondary Deployment Prompt {prompt_index}: {prompt} =====')

    url = f"{AZURE_OPENAI_ENDPOINT}/openai/deployments/{AZURE_OPENAI_SECONDARY_DEPLOYMENT}/chat/completions?api-version={AZURE_OPENAI_API_VERSION}"
    headers = {
        'api-key': AZURE_OPENAI_API_KEY,
        'Content-Type': 'application/json'
    }
    payload = {
        'messages': [
            {'role': 'system', 'content': 'You are a concise assistant.'},
            {'role': 'user', 'content': prompt}
        ],
        'max_tokens': 120
    }

    response = requests.post(url, headers=headers, json=payload, timeout=45)

    response_model = ''
    if response.status_code == 200:
        body = response.json()
        answer = body.get('choices', [{}])[0].get('message', {}).get('content', '')
        response_model = body.get('model', '')
        status = 'success'
    else:
        answer = response.text[:600]
        status = f"error_{response.status_code}"

    row = {
        'prompt_index': prompt_index,
        'query': prompt,
        'target': 'direct-secondary',
        'label': f'direct:{AZURE_OPENAI_SECONDARY_DEPLOYMENT}',
        'deployment': AZURE_OPENAI_SECONDARY_DEPLOYMENT,
        'response': answer,
        'model': response_model,
        'status': status
    }
    results.append(row)

    print(f'Status: {status}')
    print(f'Deployment: {AZURE_OPENAI_SECONDARY_DEPLOYMENT}')
    print(f"Model: {response_model if response_model else '[not returned]'}")
    print('Output:')
    print(answer.strip() if answer else '[empty response]')

## Test APIM Gateway Primary

In [ ]:
import requests

if not APIM_API_KEY or APIM_API_KEY.startswith('[Retrieve from Azure Portal'):
    raise RuntimeError('APIM_API_KEY is not set with a real key in env/.env')

for prompt_index, prompt in enumerate(test_prompts, start=1):
    print(f'\n===== APIM Primary Prompt {prompt_index}: {prompt} =====')

    url = f"{APIM_ENDPOINT}{APIM_PATH}"
    headers = {
        'Ocp-Apim-Subscription-Key': APIM_API_KEY,
        'Content-Type': 'application/json'
    }
    payload = {
        'messages': [
            {'role': 'system', 'content': 'You are a concise assistant.'},
            {'role': 'user', 'content': prompt}
        ],
        'max_tokens': 120
    }

    response = requests.post(url, headers=headers, json=payload, timeout=45)

    response_model = ''
    if response.status_code == 200:
        body = response.json()
        answer = body.get('choices', [{}])[0].get('message', {}).get('content', '')
        response_model = body.get('model', '')
        status = 'success'
    else:
        answer = response.text[:600]
        status = f"error_{response.status_code}"

    row = {
        'prompt_index': prompt_index,
        'query': prompt,
        'target': 'apim-primary',
        'label': 'apim:gateway-to-primary',
        'deployment': AZURE_OPENAI_DEPLOYMENT,
        'response': answer,
        'model': response_model,
        'status': status
    }
    results.append(row)

    print(f'Status: {status}')
    print(f'Deployment: {AZURE_OPENAI_DEPLOYMENT}')
    print(f"Model: {response_model if response_model else '[not returned]'}")
    print('Output:')
    print(answer.strip() if answer else '[empty response]')

## Test APIM Gateway Secondary

In [ ]:
import requests

if not APIM_API_KEY or APIM_API_KEY.startswith('[Retrieve from Azure Portal'):
    raise RuntimeError('APIM_API_KEY is not set with a real key in env/.env')

for prompt_index, prompt in enumerate(test_prompts, start=1):
    print(f'\n===== APIM Secondary Prompt {prompt_index}: {prompt} =====')

    url = f"{APIM_ENDPOINT}{APIM_SECONDARY_PATH}"
    headers = {
        'Ocp-Apim-Subscription-Key': APIM_API_KEY,
        'Content-Type': 'application/json'
    }
    payload = {
        'messages': [
            {'role': 'system', 'content': 'You are a concise assistant.'},
            {'role': 'user', 'content': prompt}
        ],
        'max_tokens': 120
    }

    response = requests.post(url, headers=headers, json=payload, timeout=45)

    response_model = ''
    if response.status_code == 200:
        body = response.json()
        answer = body.get('choices', [{}])[0].get('message', {}).get('content', '')
        response_model = body.get('model', '')
        status = 'success'
    else:
        answer = response.text[:600]
        status = f"error_{response.status_code}"

    row = {
        'prompt_index': prompt_index,
        'query': prompt,
        'target': 'apim-secondary',
        'label': 'apim:gateway-to-secondary',
        'deployment': AZURE_OPENAI_SECONDARY_DEPLOYMENT,
        'response': answer,
        'model': response_model,
        'status': status
    }
    results.append(row)

    print(f'Status: {status}')
    print(f'Deployment: {AZURE_OPENAI_SECONDARY_DEPLOYMENT}')
    print(f"Model: {response_model if response_model else '[not returned]'}")
    print('Output:')
    print(answer.strip() if answer else '[empty response]')

## Save Combined Results

In [ ]:
results_path = Path('../outputs/test-results.json')
results_path.parent.mkdir(parents=True, exist_ok=True)
results_path.write_text(json.dumps(results, indent=2), encoding='utf-8')

success_count = sum(1 for r in results if r['status'] == 'success')
print(f'\nCompleted checks: {success_count}/{len(results)} succeeded')
print(f'Results file: {results_path}')